# Activation function study (Table 5)

Compares five activation functions (tanh, Swish, sigmoid, ReLU and Leaky ReLU) for the selected network of 4 hidden layers with 40 neurons. Each activation is trained with three random seeds and compared with the same finite-difference reference. This study used 5000 Adam epochs.

In [ ]:
####### Imports and project directory #######
#############################################

import os
import gc
import json
import time
import random
import shutil
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import tensorflow as tf

PROJECT_DIR = Path(".")  # repository root (relative paths)

os.chdir(PROJECT_DIR)

PINN_NOTEBOOK = PROJECT_DIR / "CA_PINN_Solver.ipynb"
FD_NOTEBOOK = PROJECT_DIR / "FD_Solver.ipynb"
VALIDATION_FILE = PROJECT_DIR / "validation_utils.py"

for required_file in [
    PINN_NOTEBOOK,
    FD_NOTEBOOK,
    VALIDATION_FILE,
]:
    if not required_file.exists():
        raise FileNotFoundError(
            f"Required file not found: {required_file}"
        )

from validation_utils import rel_l2

print("Working directory:", Path.cwd())
print("TensorFlow version:", tf.__version__)
print("GPU devices:", tf.config.list_physical_devices("GPU"))

In [ ]:
######## Reproducibility settings ########
##########################################

os.environ["TF_DETERMINISTIC_OPS"] = "1"

try:
    tf.config.experimental.enable_op_determinism()
    print("TensorFlow deterministic operations enabled.")
except Exception as exc:
    print(
        "Could not explicitly enable deterministic operations:",
        repr(exc),
    )

def set_all_seeds(seed: int) -> None:
    """Reset Python, NumPy, and TensorFlow random states."""
    tf.keras.backend.clear_session()
    gc.collect()

    os.environ["PYTHONHASHSEED"] = str(seed)

    random.seed(seed)
    np.random.seed(seed)
    tf.random.set_seed(seed)

In [ ]:
# ============================================================
# EXECUTION SWITCHES
# ============================================================

# First test the notebook with TEST_MODE = True.
# This runs only Tanh with seed 42.
TEST_MODE = False

# False: preserve completed runs and skip them.
# True: rerun and replace completed runs.
OVERWRITE_EXISTING = False

# Generate a new common FD reference.
# Set this to False after the reference has been generated.
RUN_FD_REFERENCE = False

# Train the PINN activation cases.
RUN_PINN_CASES = False

# ============================================================
# BASELINE PHYSICAL PARAMETERS
# ============================================================

BASE_P0 = 1.0e21
BASE_PHI = 0.10
BASE_WCONS = 100.0

# ============================================================
# FIXED ARCHITECTURE AND TRAINING SETTINGS
# ============================================================

FIXED_HIDDEN_LAYERS = 4
FIXED_NEURONS = 40

NUM_EPOCH_ADAM = 5000
NUM_EPOCH_LBFGS = 10000
RUN_LBFGS = True

# ============================================================
# ACTIVATIONS AND RANDOM SEEDS
# ============================================================

FULL_ACTIVATIONS = [
    "tanh",
    "swish",
    "sigmoid",
    "relu",
    "leaky_relu",
]

ACTIVATION_LABELS = {
    "tanh": "Tanh",
    "swish": "Swish",
    "sigmoid": "Sigmoid",
    "relu": "ReLU",
    "leaky_relu": "Leaky ReLU",
}

FULL_SEEDS = [42, 1234, 2024]

if TEST_MODE:
    ACTIVATIONS = ["tanh", "leaky_relu"]
    SEEDS = [42]
else:
    ACTIVATIONS = FULL_ACTIVATIONS
    SEEDS = FULL_SEEDS

# ============================================================
# OUTPUT DIRECTORIES
# ============================================================

OUTPUT_ROOT = (
    PROJECT_DIR
    / "results"
    / "activation"
)

FD_REFERENCE_DIR = OUTPUT_ROOT / "fd_reference"
RUNS_DIR = OUTPUT_ROOT / "runs"
FIGURES_DIR = OUTPUT_ROOT / "figures"

for directory in [
    OUTPUT_ROOT,
    FD_REFERENCE_DIR,
    RUNS_DIR,
    FIGURES_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

RUNS_CSV = OUTPUT_ROOT / "activation_all_runs.csv"
SUMMARY_CSV = OUTPUT_ROOT / "activation_mean_std.csv"

print("Test mode:", TEST_MODE)
print("Fixed architecture:", f"{FIXED_HIDDEN_LAYERS}x{FIXED_NEURONS}")
print("Activations:", ACTIVATIONS)
print("Seeds:", SEEDS)
print("Number of PINN runs:", len(ACTIVATIONS) * len(SEEDS))
print("Output root:", OUTPUT_ROOT)

In [ ]:
### Verify activation names before starting the sweep ###
########################################################

for activation in ACTIVATIONS:
    try:
        resolved = tf.keras.activations.get(activation)
        print(f"{activation:12s} -> {resolved}")
    except Exception as exc:
        raise ValueError(
            f"TensorFlow does not recognize activation '{activation}'."
        ) from exc

In [ ]:
##### Helper functions #####
############################

REQUIRED_PINN_FILES = [
    "pinn_psi.csv",
    "pinn_n.csv",
    "pinn_I.csv",
    "loss_history.csv",
]

REQUIRED_FD_FILES = [
    "fd_psi.csv",
    "fd_n.csv",
    "fd_I.csv",
]

LEFTOVER_GLOBALS = [
    "prebias_model",
    "main_model",
    "model",
    "layers",
    "results",
    "X_0",
    "X_r",
    "X_a",
    "X_c",
    "u_a",
    "u_c",
    "X_r_pre",
    "X_0_pre",
    "X_aPre",
    "X_cPre",
    "lb",
    "ub",
    "lb_pre",
    "ub_pre",
    "lb_main",
    "ub_main",
    "u0_interp_dict",
    "u_0",
    "x_0",
    "X_test",
    "X_list",
    "T_list",
    "psi_pred",
    "n_pred",
    "I_pred",
]

def clear_solver_globals() -> None:
    """Remove variables left by previous notebook executions."""
    namespace = globals()

    for name in LEFTOVER_GLOBALS:
        namespace.pop(name, None)

    tf.keras.backend.clear_session()
    gc.collect()

def require_files(directory: Path, filenames: list[str]) -> None:
    """Raise an error if expected outputs are missing."""
    missing = [
        filename
        for filename in filenames
        if not (directory / filename).exists()
    ]

    if missing:
        raise FileNotFoundError(
            f"Missing files in {directory}: {missing}"
        )

def copy_fd_reference(case_dir: Path) -> None:
    """Copy the common FD reference into one activation-case folder."""
    require_files(FD_REFERENCE_DIR, REQUIRED_FD_FILES)

    for filename in REQUIRED_FD_FILES:
        shutil.copy2(
            FD_REFERENCE_DIR / filename,
            case_dir / filename,
        )

def compute_case_metrics(case_dir: Path) -> dict:
    """Compute validation metrics without generating comparison plots."""
    require_files(
        case_dir,
        REQUIRED_PINN_FILES + REQUIRED_FD_FILES,
    )

    pinn_psi = pd.read_csv(case_dir / "pinn_psi.csv")
    pinn_n = pd.read_csv(case_dir / "pinn_n.csv")
    pinn_I = pd.read_csv(case_dir / "pinn_I.csv")

    fd_psi = pd.read_csv(case_dir / "fd_psi.csv")
    fd_n = pd.read_csv(case_dir / "fd_n.csv")
    fd_I = pd.read_csv(case_dir / "fd_I.csv")

    for name, pinn_df, fd_df in [
        ("psi", pinn_psi, fd_psi),
        ("n", pinn_n, fd_n),
        ("I", pinn_I, fd_I),
    ]:
        if len(pinn_df) != len(fd_df):
            raise ValueError(
                f"{name}: PINN and FD grid lengths differ."
            )

        max_dx = np.max(
            np.abs(
                pinn_df["x_nm"].to_numpy()
                - fd_df["x_nm"].to_numpy()
            )
        )

        if max_dx > 1.0e-9:
            raise ValueError(
                f"{name}: PINN and FD grids differ. "
                f"Maximum difference = {max_dx:.3e} nm."
            )

    I_mean_pinn = float(pinn_I["I_m3"].mean())
    I_mean_fd = float(fd_I["I_m3"].mean())

    metrics = {
        "relL2_psi": float(rel_l2(pinn_psi["psi_V"], fd_psi["psi_V"])),
        "relL2_n": float(rel_l2(pinn_n["n_m3"], fd_n["n_m3"])),
        "relL2_I": float(rel_l2(pinn_I["I_m3"], fd_I["I_m3"])),
        "I_mean_PINN": I_mean_pinn,
        "I_mean_FD": I_mean_fd,
        "I_mean_rel_error": (
            abs(I_mean_pinn - I_mean_fd)
            / (abs(I_mean_fd) + 1.0e-300)
        ),
    }

    # Retained as an internal diagnostic only.
    metrics["aggregate_error"] = float(
        np.exp(
            np.mean(
                np.log(
                    np.maximum(
                        [
                            metrics["relL2_psi"],
                            metrics["relL2_n"],
                            metrics["relL2_I"],
                        ],
                        1.0e-300,
                    )
                )
            )
        )
    )

    loss_df = pd.read_csv(
        case_dir / "loss_history.csv"
    )

    if "total_loss" not in loss_df.columns:
        raise ValueError(
            "The loss-history file does not contain a 'total_loss' column."
        )

    metrics["final_total_loss"] = float(
        loss_df["total_loss"].iloc[-1]
    )

    return metrics

def count_trainable_parameters() -> int:
    """Count parameters in the trained PINN."""
    if "prebias_model" not in globals():
        raise RuntimeError(
            "prebias_model is unavailable after PINN execution."
        )

    variables = prebias_model.model.trainable_variables

    return int(
        sum(np.prod(variable.shape) for variable in variables)
    )

In [ ]:
### Generate the common finite-difference reference once ###
###########################################################

if RUN_FD_REFERENCE:
    print("=" * 80)
    print("Generating the common FD reference")
    print("=" * 80)

    clear_solver_globals()

    globals()["P0"] = BASE_P0
    globals()["phia"] = BASE_PHI
    globals()["phic"] = BASE_PHI
    globals()["OUTPUT_DIR"] = str(FD_REFERENCE_DIR)

    fd_start = time.time()

    get_ipython().run_line_magic(
        "run",
        f'-i "{FD_NOTEBOOK}"',
    )

    fd_runtime = time.time() - fd_start

    require_files(FD_REFERENCE_DIR, REQUIRED_FD_FILES)

    print(f"FD reference completed in {fd_runtime:.1f} s.")

else:
    require_files(FD_REFERENCE_DIR, REQUIRED_FD_FILES)
    print("Using the existing FD reference in:", FD_REFERENCE_DIR)

In [ ]:
### Function for one activation and seed ###
###########################################

def run_activation_case(activation: str, seed: int) -> dict:
    """Train and validate one activation function for one seed."""

    activation_label = ACTIVATION_LABELS[activation]
    safe_name = activation.replace("_", "-")

    tag = (
        f"L{FIXED_HIDDEN_LAYERS}_W{FIXED_NEURONS}"
        f"_{safe_name}_seed{seed}"
    )

    case_dir = RUNS_DIR / tag
    case_dir.mkdir(parents=True, exist_ok=True)

    completed_files = REQUIRED_PINN_FILES + REQUIRED_FD_FILES

    already_complete = all(
        (case_dir / filename).exists()
        for filename in completed_files
    )

    if already_complete and not OVERWRITE_EXISTING:
        print(f"\nSkipping completed case: {tag}")

        metrics = compute_case_metrics(case_dir)

        return {
            "activation": activation,
            "activation_label": activation_label,
            "hidden_layers": FIXED_HIDDEN_LAYERS,
            "neurons": FIXED_NEURONS,
            "architecture": f"{FIXED_HIDDEN_LAYERS}x{FIXED_NEURONS}",
            "seed": seed,
            "P0": BASE_P0,
            "phi": BASE_PHI,
            "W_cons": BASE_WCONS,
            "num_epoch_adam": NUM_EPOCH_ADAM,
            "num_epoch_lbfgs": NUM_EPOCH_LBFGS,
            "run_lbfgs": RUN_LBFGS,
            "training_runtime_s": np.nan,
            "trainable_parameters": np.nan,
            **metrics,
            "status": "reused",
            "case_dir": str(case_dir),
        }

    print("\n" + "=" * 80)
    print(f"Running activation case: {tag}")
    print(f"Activation: {activation_label}")
    print(f"Architecture: {FIXED_HIDDEN_LAYERS}x{FIXED_NEURONS}")
    print(f"Seed: {seed}")
    print(f"Output: {case_dir}")
    print("=" * 80)

    clear_solver_globals()
    set_all_seeds(seed)

    # Parameters consumed by CA_PINN_Solver.ipynb
    globals()["P0"] = BASE_P0
    globals()["phia"] = BASE_PHI
    globals()["phic"] = BASE_PHI

    globals()["HIDDEN_LAYERS"] = FIXED_HIDDEN_LAYERS
    globals()["NEURONS"] = FIXED_NEURONS
    globals()["ACTIVATION"] = activation

    globals()["W_cons"] = BASE_WCONS

    globals()["NUM_EPOCH_ADAM"] = NUM_EPOCH_ADAM
    globals()["NUM_EPOCH_LBFGS"] = NUM_EPOCH_LBFGS
    globals()["RUN_LBFGS"] = RUN_LBFGS

    globals()["OUTPUT_DIR"] = str(case_dir)

    training_start = time.time()

    try:
        if RUN_PINN_CASES:
            get_ipython().run_line_magic(
                "run",
                f'-i "{PINN_NOTEBOOK}"',
            )

        training_runtime = time.time() - training_start

        require_files(case_dir, REQUIRED_PINN_FILES)

        trainable_parameters = count_trainable_parameters()

        copy_fd_reference(case_dir)
        metrics = compute_case_metrics(case_dir)

        row = {
            "activation": activation,
            "activation_label": activation_label,
            "hidden_layers": FIXED_HIDDEN_LAYERS,
            "neurons": FIXED_NEURONS,
            "architecture": f"{FIXED_HIDDEN_LAYERS}x{FIXED_NEURONS}",
            "seed": seed,
            "P0": BASE_P0,
            "phi": BASE_PHI,
            "W_cons": BASE_WCONS,
            "num_epoch_adam": NUM_EPOCH_ADAM,
            "num_epoch_lbfgs": NUM_EPOCH_LBFGS,
            "run_lbfgs": RUN_LBFGS,
            "training_runtime_s": float(training_runtime),
            "trainable_parameters": trainable_parameters,
            **metrics,
            "status": "ok",
            "case_dir": str(case_dir),
        }

        print(
            f"[{tag}] relative L2 errors: "
            f"psi={metrics['relL2_psi']:.3e}, "
            f"n={metrics['relL2_n']:.3e}, "
            f"I={metrics['relL2_I']:.3e}"
        )

        return row

    except Exception as exc:
        print(f"[{tag}] FAILED: {repr(exc)}")

        return {
            "activation": activation,
            "activation_label": activation_label,
            "hidden_layers": FIXED_HIDDEN_LAYERS,
            "neurons": FIXED_NEURONS,
            "architecture": f"{FIXED_HIDDEN_LAYERS}x{FIXED_NEURONS}",
            "seed": seed,
            "P0": BASE_P0,
            "phi": BASE_PHI,
            "W_cons": BASE_WCONS,
            "num_epoch_adam": NUM_EPOCH_ADAM,
            "num_epoch_lbfgs": NUM_EPOCH_LBFGS,
            "run_lbfgs": RUN_LBFGS,
            "training_runtime_s": float(time.time() - training_start),
            "trainable_parameters": np.nan,
            "relL2_psi": np.nan,
            "relL2_n": np.nan,
            "relL2_I": np.nan,
            "I_mean_PINN": np.nan,
            "I_mean_FD": np.nan,
            "I_mean_rel_error": np.nan,
            "aggregate_error": np.nan,
            "final_total_loss": np.nan,
            "status": f"failed: {type(exc).__name__}",
            "case_dir": str(case_dir),
        }

In [ ]:
### Run the activation-function sweep ###
#########################################

if RUNS_CSV.exists() and not OVERWRITE_EXISTING:
    existing_df = pd.read_csv(RUNS_CSV)
    existing_rows = existing_df.to_dict("records")
else:
    existing_rows = []

existing_keys = {
    (str(row["activation"]), int(row["seed"]))
    for row in existing_rows
    if str(row.get("status", "")).startswith(("ok", "reused"))
}

rows = existing_rows.copy()

for activation in ACTIVATIONS:
    for seed in SEEDS:
        key = (activation, seed)

        if key in existing_keys and not OVERWRITE_EXISTING:
            print("Already recorded, skipping:", key)
            continue

        row = run_activation_case(
            activation=activation,
            seed=seed,
        )

        rows.append(row)

        pd.DataFrame(rows).to_csv(
            RUNS_CSV,
            index=False,
        )

        print("Updated:", RUNS_CSV)

print("\nActivation-function sweep completed.")

In [ ]:
### Inspect all individual runs ###
###################################

runs_df = pd.read_csv(RUNS_CSV)

activation_order = FULL_ACTIVATIONS
runs_df["activation"] = pd.Categorical(
    runs_df["activation"],
    categories=activation_order,
    ordered=True,
)

display_columns = [
    "activation_label",
    "seed",
    "relL2_psi",
    "relL2_n",
    "relL2_I",
    "I_mean_rel_error",
    "training_runtime_s",
    "trainable_parameters",
    "status",
]

display(
    runs_df
    .sort_values(["activation", "seed"])[display_columns]
    .reset_index(drop=True)
)

In [ ]:
### Aggregate mean and standard deviation ###
#############################################

successful_df = runs_df[
    runs_df["status"].astype(str).str.startswith(("ok", "reused"))
].copy()

metric_columns = [
    "relL2_psi",
    "relL2_n",
    "relL2_I",
    "I_mean_rel_error",
    "aggregate_error",
    "final_total_loss",
    "training_runtime_s",
    "trainable_parameters",
]

summary_rows = []

for activation in FULL_ACTIVATIONS:
    group = successful_df[
        successful_df["activation"].astype(str) == activation
    ]

    if group.empty:
        continue

    row = {
        "activation": activation,
        "activation_label": ACTIVATION_LABELS[activation],
        "hidden_layers": FIXED_HIDDEN_LAYERS,
        "neurons": FIXED_NEURONS,
        "architecture": f"{FIXED_HIDDEN_LAYERS}x{FIXED_NEURONS}",
        "num_successful_seeds": len(group),
    }

    for metric in metric_columns:
        values = group[metric].dropna()

        row[f"{metric}_mean"] = (
            float(values.mean()) if len(values) else np.nan
        )

        row[f"{metric}_std"] = (
            float(values.std(ddof=1)) if len(values) > 1 else np.nan
        )

    summary_rows.append(row)

summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(SUMMARY_CSV, index=False)

print("Saved:", SUMMARY_CSV)

display_columns = [
    "activation_label",
    "num_successful_seeds",
    "relL2_psi_mean",
    "relL2_psi_std",
    "relL2_n_mean",
    "relL2_n_std",
    "relL2_I_mean",
    "relL2_I_std",
    "training_runtime_s_mean",
    "training_runtime_s_std",
]

display(summary_df[display_columns])

In [ ]:
### Plot total-loss histories for all successful runs ###
########################################################

fig, ax = plt.subplots(figsize=(7.6, 5.2))

for activation in FULL_ACTIVATIONS:
    activation_runs = successful_df[
        successful_df["activation"].astype(str) == activation
    ]

    for run_index, (_, row) in enumerate(activation_runs.iterrows()):
        loss_path = Path(row["case_dir"]) / "loss_history.csv"

        if not loss_path.exists():
            continue

        loss_df = pd.read_csv(loss_path)

        label = (
            ACTIVATION_LABELS[activation]
            if run_index == 0
            else None
        )

        ax.plot(
            loss_df["epoch"],
            loss_df["total_loss"],
            linewidth=1.3,
            alpha=0.75,
            label=label,
        )

ax.axvline(
    x=NUM_EPOCH_ADAM,
    linestyle="--",
    linewidth=1.3,
    label="Adam to L-BFGS",
)

ax.set_yscale("log")
ax.set_xlabel("Epoch")
ax.set_ylabel("Total loss")
ax.grid(True, which="both", alpha=0.25)
ax.legend(frameon=False)

fig.tight_layout()

loss_png = FIGURES_DIR / "activation_total_loss_histories.png"
loss_pdf = FIGURES_DIR / "activation_total_loss_histories.pdf"

fig.savefig(loss_png, dpi=600, bbox_inches="tight")
fig.savefig(loss_pdf, bbox_inches="tight")
plt.show()

print("Saved:", loss_png)
print("Saved:", loss_pdf)